# Assignment 7 -- Pareto Front Visualisation & Regional Pathways

**Course:** EPA141A Model-Based Decision Making -- Delft University of Technology  
**Model:** JUSTICE  

> **Pre-requisite:** Assignment 6 (MOEA Convergence & Reference Set). The `reference_set_utilitarian.csv` saved in Assignment 6 is required.

---

## Learning Outcomes

After completing this assignment you will be able to:

1. Read and discuss **policy trade-offs** from a parallel-coordinates visualisation.
2. Identify the most conflicting objective pairs from a scatter plot.
3. Interpret **regionalised emission control pathways** on a global map.
4. Connect Pareto front insights to actor perspectives and the XLRM framework.

---

## Background

### Trade-off analysis

A solution is **Pareto-optimal** if you cannot improve one objective without worsening another. The **parallel-coordinates plot** is the standard visualisation for exploring trade-offs across 4+ objectives simultaneously:

- Each vertical axis represents one objective.
- Each line represents one non-dominated policy.
- A **crossing** between two adjacent axes reveals a trade-off.
- Axes are oriented so that **up = better** on every axis.

### Regional emission control rates

The JUSTICE model assigns an **emission control rate (ECR)** to each of the 57 RICE50 world regions. ECR ranges from 0 (no abatement) to 1 (full decarbonisation). Visualising the time-averaged ECR on a world map reveals **which regions are expected to decarbonise most aggressively** under each policy.

---

## Overview

Starting from the reference set saved in Assignment 6, you will:

1. **Visualise the Pareto front** with a parallel-coordinates plot showing all four objectives.
2. **Map regional ECR pathways** by re-running JUSTICE with four anchor policies and plotting the time-averaged ECR per RICE50 region on a world map.

## Setup -- Imports and configuration

The cell below imports all required packages, applies the Python 3.14 compatibility patch for `matplotlib.path.Path`, and loads the cross-seed reference set produced in Assignment 6.

In [ ]:
# Standard imports
import warnings
warnings.filterwarnings("ignore")

import os, sys, json, glob, copy
import numpy as np
import pandas as pd

# Matplotlib deepcopy patch (Python 3.14 + matplotlib compatibility)
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# DEAP hypervolume
from deap.tools._hypervolume import hv as deap_hv

# EMA Workbench
from ema_workbench.em_framework.optimization import epsilon_nondominated
from ema_workbench import RealParameter, ScalarOutcome
from ema_workbench.em_framework.optimization import Problem

# Geopandas for maps
import geopandas as gpd

# Path setup
_NOTEBOOK_DIR = os.path.abspath(".")
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

# Objective metadata
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2 C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
}

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_ROOT}")
print(f"DEAP hv      : OK")
print(f"geopandas    : {gpd.__version__}")
print("Matplotlib deepcopy patch applied.")

In [ ]:
# Load reference set from Assignment 6
ref_path = os.path.join(RESULTS_ROOT, "reference_set_utilitarian.csv")
if not os.path.exists(ref_path):
    raise FileNotFoundError(
        f"Reference set not found: {ref_path}\n"
        "Run Assignment 6 (Steps 1-3) first to build and save the reference set."
    )

ref_set = pd.read_csv(ref_path)
ref_set = ref_set[ref_set["welfare"] < 1e5].reset_index(drop=True)

print(f"Reference set loaded: {len(ref_set)} solutions x {len(ref_set.columns)} columns")
print(f"\nObjective statistics:")
print(ref_set[OBJECTIVE_COLS].describe().round(3).to_string())

---

## Step 1 -- Pareto Front Visualisation & Trade-off Discussion

Two complementary plots reveal the structure of the reference set:

1. **Parallel-coordinates plot** of all four objectives across the reference set.  
   Each line is one policy; axes are oriented so that **up = better** on every axis.  
   Lines are coloured by `fraction_above_threshold` (climate risk).  
   Crossings between adjacent axes reveal **trade-offs**.

**Task 1.1** -- Run the cell below to generate the parallel-coordinates plot.

**Task 1.2** -- Identify the two axes that show the clearest crossing pattern (strongest trade-off). Write your answer in the reflection questions.

In [ ]:
# Normalise objectives for parallel-coordinates
# For MINIMIZE: invert so up = better; for MAXIMIZE: scale so up = better
obj = ref_set[OBJECTIVE_COLS].copy()

def normalise_for_plot(df_obj):
    normed = df_obj.copy().astype(float)
    for col in MINIMIZE_COLS:
        lo, hi = normed[col].min(), normed[col].max()
        normed[col] = 1.0 - (normed[col] - lo) / (hi - lo + 1e-15)
    for col in MAXIMIZE_COLS:
        lo, hi = normed[col].min(), normed[col].max()
        normed[col] = (normed[col] - lo) / (hi - lo + 1e-15)
    return normed

obj_norm = normalise_for_plot(obj)

# Identify four anchor policies (one per extreme objective)
extremes = {
    "Min climate risk":   obj["fraction_above_threshold"].idxmin(),
    "Min abatement cost": obj["welfare_loss_abatement"].idxmin(),
    "Best welfare":       obj["welfare"].idxmin(),
    "Max damage avoided": obj["welfare_loss_damage"].idxmax(),
}
extreme_colors = ["#1565C0", "#B71C1C", "#2E7D32", "#E65100"]

fig, ax = plt.subplots(figsize=(11, 6))
x_pos = np.arange(len(OBJECTIVE_COLS))
axis_labels = [OBJECTIVE_LABELS[c] for c in OBJECTIVE_COLS]

frac_vals = obj_norm["fraction_above_threshold"].values
cmap_risk  = plt.cm.RdYlBu_r
norm_risk  = Normalize(vmin=frac_vals.min(), vmax=frac_vals.max())

# Draw all reference-set solutions coloured by climate risk
# Hint: for idx, row in obj_norm.iterrows():
#           y = row[OBJECTIVE_COLS].values
#           color = cmap_risk(norm_risk(row["fraction_above_threshold"]))
#           ax.plot(x_pos, y, color=color, lw=1.4, alpha=0.55, zorder=2)
...

# Overlay four extreme policies with thick coloured lines
# Hint: for (label, idx), ec in zip(extremes.items(), extreme_colors):
#           ax.plot(x_pos, obj_norm.loc[idx, OBJECTIVE_COLS].values, color=ec, lw=3.2, ...)
...

# Axis styling (provided)
for xi in x_pos:
    ax.axvline(xi, color="0.75", lw=0.8, zorder=1)
ax.set_xticks(x_pos)
ax.set_xticklabels(axis_labels, fontsize=10)
ax.set_ylabel("Normalised performance  (up = better)", fontsize=10)
ax.set_ylim(-0.08, 1.12)
ax.set_title(
    "Parallel-Coordinates Plot — JUSTICE Reference Set\n"
    "(all axes: up = better; colour = climate risk — darker red = higher risk)",
    fontsize=11,
)

# Add a colourbar for climate risk and a legend for the four anchor policies
...

plt.tight_layout()
plt.savefig("parcoords_reference_set.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: parcoords_reference_set.png")

### Trade-off Discussion

**Task 1.3** -- Answer the following questions based on your parallel-coordinates plot.

- **Welfare vs. Climate risk:** What pattern do you observe between the welfare axis and the climate risk axis? Is there a clear trade-off or do they co-vary?

*(Write your answer here.)*

- **Climate risk vs. Abatement cost:** Describe the crossing pattern between these two axes. What does this imply in physical/economic terms?

*(Write your answer here.)*

- **Policy anchors:** How do the four coloured extreme policies differ from one another on the parallel-coordinates plot? Which axis separates them most clearly?

*(Write your answer here.)*

> **Student task:** Describe the trade-off you observe between `welfare_loss_damage` and `welfare_loss_abatement`. Do policies with high abatement cost always achieve proportionally lower damage? What does this imply for your actor?

---

## Step 2 -- Regional Emission Control Rate (ECR) Pathways

The RBF policy maps real-time climate state (temperature, warming rate) to an **emission control rate** for each of the 57 RICE50 regions.  
We re-run the JUSTICE model for the four best-performing policies on the Pareto front, each anchored to one objective:

| Label | Anchor | Direction |
|---|---|---|
| **W** | Maximum welfare | Highest welfare score |
| **R** | Minimum climate risk | Lowest fraction of scenarios above 2 C |
| **D** | Minimum damage cost | Lowest welfare loss from climate damage |
| **Ab** | Minimum abatement cost | Lowest welfare loss from abatement |

We then visualise the **time-averaged ECR** on world choropleth maps, one per anchor.

> **Note:** We use 1 FAIR ensemble member here for speed. The policy RBF parameters come directly from the reference-set CSV.

In [ ]:
# JUSTICE imports and model constants
import warnings
warnings.filterwarnings("ignore")

from justice.model import JUSTICE
from justice.util.data_loader import DataLoader
from justice.util.enumerations import (
    Abatement, DamageFunction, Economy, WelfareFunction
)
from justice.util.emission_control_constraint import EmissionControlConstraint
from justice.util.model_time import TimeHorizon
from solvers.emodps.rbf import RBF

# Model constants (from config/config_ssp245.json)
with open(os.path.join(_NOTEBOOK_DIR, "../config/config_ssp245.json")) as fh:
    _cfg = json.load(fh)

_time_horizon = TimeHorizon(
    start_year            = _cfg["start_year"],
    end_year              = _cfg["end_year"],
    data_timestep         = _cfg["data_timestep"],
    timestep              = _cfg["timestep"],
)
N_TIMESTEPS    = len(_time_horizon.model_time_horizon)
N_REGIONS      = len(DataLoader().REGION_LIST)
REGION_LIST    = list(DataLoader().REGION_LIST)   # 57 RICE50 region codes
N_INPUTS_RBF   = _cfg["n_inputs"]          # 2
N_RBFS         = _cfg["n_inputs"] + 2      # 4
SCENARIO       = _cfg["reference_ssp_rcp_scenario_index"]   # 2 = SSP2-RCP4.5
EC_START_TS    = _time_horizon.year_to_timestep(
    year=_cfg["emission_control_start_year"],
    timestep=_cfg["timestep"],
)
_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF = 2.0,  0.0

# Helper: run JUSTICE for one policy row, return ECR array
def run_policy_ecr(policy_row, n_ensemble=1):
    """
    Run JUSTICE stepwise with the given RBF policy parameters.

    Parameters
    ----------
    policy_row : pd.Series  -- one row from the reference set
    n_ensemble : int        -- number of FAIR ensemble members to use

    Returns
    -------
    ecr_mean : np.ndarray shape (N_REGIONS, N_TIMESTEPS)
        Mean constrained ECR across ensemble members.
    datasets : dict  -- final JUSTICE evaluate() output
    """
    # Build RBF from policy parameters
    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS_RBF, n_outputs=N_REGIONS)
    c_shape, r_shape, w_shape = rbf.get_shape()

    centers = np.array([policy_row[f"center {i}"] for i in range(c_shape[0])])
    radii   = np.array([policy_row[f"radii {i}"]  for i in range(r_shape[0])])
    weights = np.array([policy_row[f"weights {i}"] for i in range(w_shape[0])])
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    ensemble_indices = list(np.linspace(1, 1000, max(n_ensemble, 10), dtype=int))[:n_ensemble]

    model = JUSTICE(
        scenario=SCENARIO,
        climate_ensembles=ensemble_indices,
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function_type=WelfareFunction.UTILITARIAN.value[0],
    )
    no_ens = model.no_of_ensembles

    ecr             = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = 0.0
    diff      = 0.0

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )
        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data = model.stepwise_evaluate(timestep=t)
        temp = data["global_temperature"][t, :]

        if t % 5 == 0:
            diff      = temp - prev_temp
            prev_temp = temp

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    datasets = model.evaluate()
    ecr_mean = constrained_ecr.mean(axis=2)
    return ecr_mean, datasets


# Select four anchor policies from the reference set
obj_ref = ref_set[OBJECTIVE_COLS]

idx_W  = obj_ref["welfare"].idxmax()
idx_R  = obj_ref["fraction_above_threshold"].idxmin()
idx_D  = obj_ref["welfare_loss_damage"].idxmin()
idx_Ab = obj_ref["welfare_loss_abatement"].idxmin()

anchors = {
    "W  -- Max welfare":        idx_W,
    "R  -- Min climate risk":   idx_R,
    "D  -- Min damage cost":    idx_D,
    "Ab -- Min abatement cost": idx_Ab,
}

print("Anchor policies selected from Pareto reference set:\n")
for label, idx in anchors.items():
    print(f"  {label}")
    print(obj_ref.loc[idx].round(3).to_string(index=True))
    print()

print("Running JUSTICE model for all four anchor policies ...")
ecr_W,  _ = run_policy_ecr(ref_set.loc[idx_W],  n_ensemble=1)
ecr_R,  _ = run_policy_ecr(ref_set.loc[idx_R],  n_ensemble=1)
ecr_D,  _ = run_policy_ecr(ref_set.loc[idx_D],  n_ensemble=1)
ecr_Ab, _ = run_policy_ecr(ref_set.loc[idx_Ab], n_ensemble=1)
print("Done.")

In [ ]:
# RICE50 region -> world map country names (provided -- do not modify)
RICE50_TO_GEO = {
    "arg": ["Argentina"], "aus": ["Australia"], "aut": ["Austria"],
    "bel": ["Belgium"], "bgr": ["Bulgaria"], "blt": ["Estonia", "Latvia", "Lithuania"],
    "bra": ["Brazil"], "can": ["Canada"], "chl": ["Chile"], "chn": ["China"],
    "cor": ["South Korea"], "cro": ["Croatia"], "dnk": ["Denmark"], "egy": ["Egypt"],
    "esp": ["Spain"], "fin": ["Finland"], "fra": ["France"], "gbr": ["United Kingdom"],
    "golf57": ["Saudi Arabia", "Kuwait", "Qatar", "United Arab Emirates", "Oman"],
    "grc": ["Greece"], "hun": ["Hungary"], "idn": ["Indonesia"], "irl": ["Ireland"],
    "ita": ["Italy"], "jpn": ["Japan"],
    "meme": ["Jordan", "Israel", "Lebanon", "Syria", "Iraq", "Iran", "Yemen"],
    "mex": ["Mexico"], "mys": ["Malaysia"], "nde": ["India"], "nld": ["Netherlands"],
    "noan": ["W. Sahara", "Tunisia", "Morocco"], "noap": ["Libya", "Algeria"],
    "nor": ["Norway"],
    "oeu": ["Albania", "Bosnia and Herz.", "N. Macedonia", "Montenegro", "Serbia"],
    "osea": ["Philippines", "Myanmar", "Cambodia", "Laos"],
    "pol": ["Poland"], "prt": ["Portugal"],
    "rcam": ["Costa Rica", "Guatemala", "Honduras", "Nicaragua", "Panama",
             "El Salvador", "Belize"],
    "rcz": ["Czechia"], "rfa": ["Germany"],
    "ris": ["Kazakhstan", "Uzbekistan", "Turkmenistan", "Tajikistan", "Kyrgyzstan",
            "Georgia", "Armenia", "Azerbaijan", "Belarus", "Moldova"],
    "rjan57": ["Papua New Guinea", "New Zealand", "Fiji"],
    "rom": ["Romania"],
    "rsaf": ["Tanzania", "Mozambique", "Zimbabwe", "Zambia", "Malawi", "Botswana",
             "Namibia", "Angola", "Nigeria", "Ghana", "Cameroon", "Ethiopia",
             "Kenya", "Uganda", "Somalia", "Sudan", "Chad", "Niger", "Mali",
             "Burkina Faso", "Senegal", "Guinea", "Ivory Coast", "Dem. Rep. Congo",
             "Congo", "Central African Rep.", "Madagascar", "Eritrea", "S. Sudan",
             "Rwanda", "Burundi", "Togo", "Benin", "Liberia", "Sierra Leone"],
    "rsam": ["Venezuela", "Colombia", "Peru", "Ecuador", "Bolivia",
             "Paraguay", "Uruguay", "Guyana", "Suriname"],
    "rsas": ["Pakistan", "Bangladesh", "Sri Lanka", "Nepal"],
    "rsl": ["Slovakia"], "rus": ["Russia"], "slo": ["Slovenia"], "sui": ["Switzerland"],
    "swe": ["Sweden"], "tha": ["Thailand"], "tur": ["Turkey"], "ukr": ["Ukraine"],
    "usa": ["United States of America"], "vnm": ["Vietnam"], "zaf": ["South Africa"],
}

# Build reverse mapping: country name -> RICE50 code
country_to_rice50 = {}
for code_key, countries in RICE50_TO_GEO.items():
    for c in countries:
        country_to_rice50[c] = code_key

# Compute time-averaged ECR per region
mean_ecr_W  = ecr_W.mean(axis=1)
mean_ecr_R  = ecr_R.mean(axis=1)
mean_ecr_D  = ecr_D.mean(axis=1)
mean_ecr_Ab = ecr_Ab.mean(axis=1)

ecr_region_W  = {REGION_LIST[i]: mean_ecr_W[i]  for i in range(N_REGIONS)}
ecr_region_R  = {REGION_LIST[i]: mean_ecr_R[i]  for i in range(N_REGIONS)}
ecr_region_D  = {REGION_LIST[i]: mean_ecr_D[i]  for i in range(N_REGIONS)}
ecr_region_Ab = {REGION_LIST[i]: mean_ecr_Ab[i] for i in range(N_REGIONS)}

# Load world shapefile
import importlib.util, pathlib
_pyogrio_path = pathlib.Path(importlib.util.find_spec("pyogrio").origin).parent
_ne_shp = str(_pyogrio_path / "tests" / "fixtures" / "naturalearth_lowres" / "naturalearth_lowres.shp")
world = gpd.read_file(_ne_shp)

world["rice50"]  = world["name"].map(country_to_rice50)
world["ecr_W"]   = world["rice50"].map(ecr_region_W)
world["ecr_R"]   = world["rice50"].map(ecr_region_R)
world["ecr_D"]   = world["rice50"].map(ecr_region_D)
world["ecr_Ab"]  = world["rice50"].map(ecr_region_Ab)

n_mapped = world["rice50"].notna().sum()
print(f"Countries mapped to RICE50 regions: {n_mapped}/{len(world)}")

In [ ]:
ecr_cols = ["ecr_W", "ecr_R", "ecr_D", "ecr_Ab"]
all_vals  = pd.concat([world[c].dropna() for c in ecr_cols])
vmin_all, vmax_all = all_vals.min(), all_vals.max()

panel_specs = [
    ("ecr_W",  "W — Max Welfare\nTime-averaged ECR per region"),
    ("ecr_R",  "R — Min Climate Risk\nTime-averaged ECR per region"),
    ("ecr_D",  "D — Min Damage Cost\nTime-averaged ECR per region"),
    ("ecr_Ab", "Ab — Min Abatement Cost\nTime-averaged ECR per region"),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, (col, title) in zip(axes.flatten(), panel_specs):
    # Plot unmapped countries in grey, then mapped countries coloured by ECR
    # Hint: world[world[col].isna()].plot(ax=ax, color="0.88", ...)
    #       world[world[col].notna()].plot(column=col, ax=ax, cmap=..., vmin=vmin_all, vmax=vmax_all, ...)
    ...
    ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
    ax.set_axis_off()

fig.suptitle(
    "Best-performing Policies on the Pareto Front — Regional ECR (JUSTICE Model)\n"
    "(grey = regions not mapped; shared colour scale across all panels)",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig("ecr_regional_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ecr_regional_map.png")

In [ ]:
df_ecr = pd.DataFrame({
    "region":              REGION_LIST,
    "W — Max welfare":     mean_ecr_W,
    "R — Min risk":        mean_ecr_R,
    "D — Min damage":      mean_ecr_D,
    "Ab — Min abatement":  mean_ecr_Ab,
}).set_index("region").sort_values("R — Min risk", ascending=False)

policy_colors = {
    "W — Max welfare":    "#2E7D32",
    "R — Min risk":       "#1565C0",
    "D — Min damage":     "#F57F17",
    "Ab — Min abatement": "#B71C1C",
}
n_policies = len(policy_colors)
w = 0.8 / n_policies
x = np.arange(len(df_ecr))

fig, ax = plt.subplots(figsize=(18, 5))
for k, (label, color) in enumerate(policy_colors.items()):
    offset = (k - (n_policies - 1) / 2) * w
    ax.bar(x + offset, df_ecr[label], width=w, color=color, alpha=0.85, label=label)

ax.set_xticks(x)
ax.set_xticklabels(df_ecr.index, rotation=90, fontsize=7.5)
ax.set_ylabel("Time-averaged ECR (0-1)")
ax.set_title(
    "Emission Control Rate per RICE50 Region — Four Best-performing Pareto Policies\n"
    "(sorted by Min Climate Risk ECR; higher ECR = more aggressive decarbonisation)",
    fontsize=11,
)
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.savefig("ecr_regional_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ecr_regional_bar.png")

---

## Reflection Questions

**1. Trade-offs on the parallel-coordinates plot.** Identify the two axes that show the clearest crossing pattern (i.e., the strongest trade-off). Explain in 2–3 sentences why this trade-off exists in physical/economic terms.


*(Write your answer here.)*

---

**2. Emission control rate map.** Compare the min-climate-risk policy and the min-abatement-cost policy on the world map. Which regions show the largest difference in ECR between the two? What does this imply about which regions are most influential in driving down climate risk?


*(Write your answer here.)*

---

**3. Actor perspective.** Based on the parallel-coordinates plot and the regional ECR map, which policy from the reference set would you recommend to your actor? Justify your choice with explicit reference to at least two objectives and one regional consideration.


*(Write your answer here.)*